# Delta Lake com PySpark

Notebook de referência para configurar Spark + Delta Lake e executar carga inicial (gravada com `overwrite`), operações `UPDATE` e `DELETE`, além de consultar o histórico de versões e *time travel* no mesmo cenário Statcast do projeto.

## Configuração do Spark com Delta Lake

**O que este bloco faz:** importa `SparkSession` e o pacote `delta`, define o `SparkSession` com as extensões SQL do Delta (`DeltaSparkSessionExtension`), associa o catálogo Spark ao `DeltaCatalog`, fixa o diretório do warehouse em `../data/warehouse` e usa `configure_spark_with_delta_pip` para alinhar os JARs do Delta ao PySpark instalado. O resultado é uma sessão Spark pronta para ler e gravar tabelas no formato Delta.

In [1]:
from pyspark.sql import SparkSession
from delta import *

builder = SparkSession.builder.appName("DeltaLake_SATC") \
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension") \
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog") \
    .config("spark.sql.warehouse.dir", "../data/warehouse")

spark = configure_spark_with_delta_pip(builder).getOrCreate()

26/04/18 14:36:15 WARN Utils: Your hostname, PC-Octavio resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
26/04/18 14:36:15 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address


:: loading settings :: url = jar:file:/mnt/c/source/trabalho1ed/.venv/lib/python3.11/site-packages/pyspark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /home/octavio/.ivy2/cache
The jars for the packages stored in: /home/octavio/.ivy2/jars
io.delta#delta-spark_2.12 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-cd3d3a52-71ae-4a0c-8aa4-147c3da26856;1.0
	confs: [default]
	found io.delta#delta-spark_2.12;3.2.0 in central
	found io.delta#delta-storage;3.2.0 in central
	found org.antlr#antlr4-runtime;4.9.3 in central
downloading https://repo1.maven.org/maven2/io/delta/delta-spark_2.12/3.2.0/delta-spark_2.12-3.2.0.jar ...
	[SUCCESSFUL ] io.delta#delta-spark_2.12;3.2.0!delta-spark_2.12.jar (502ms)
downloading https://repo1.maven.org/maven2/io/delta/delta-storage/3.2.0/delta-storage-3.2.0.jar ...
	[SUCCESSFUL ] io.delta#delta-storage;3.2.0!delta-storage.jar (258ms)
downloading https://repo1.maven.org/maven2/org/antlr/antlr4-runtime/4.9.3/antlr4-runtime-4.9.3.jar ...
	[SUCCESSFUL ] org.antlr#antlr4-runtime;4.9.3!antlr4-runtime.jar (580ms)
:: resolution report :: resolve 3510ms 

## Carga inicial da tabela Delta (Statcast)

**O que este bloco faz:** inclui a raiz do projeto no `sys.path`, importa `ler_e_limpar_dados` e `obter_esquema_statcast` de `src/ingestao.py`, lê o CSV bruto em `data/raw/statcast_data.csv`, valida o número de colunas contra o esquema, grava a tabela em formato Delta em `data/delta_statcast` com modo `overwrite` e exibe uma amostra das linhas.

Contexto: estatísticas agregadas da MLB Statcast; `ler_e_limpar_dados` aplica *cast* e renomeia colunas para o modelo **DESEMPENHO_ARREMESSADOR**. Os dados ficam em Parquet com log transacional do Delta, habilitando operações ACID e evolução controlada sem reprocessar o arquivo fonte a cada ajuste pontual.

In [2]:
import sys
from pathlib import Path

raiz_projeto = Path("..").resolve()
if str(raiz_projeto) not in sys.path:
    sys.path.insert(0, str(raiz_projeto))

from src.ingestao import obter_esquema_statcast, ler_e_limpar_dados

CAMINHO_CSV = str(raiz_projeto / "data" / "raw" / "statcast_data.csv")
CAMINHO_DELTA = str(raiz_projeto / "data" / "delta_statcast")

esquema_statcast = obter_esquema_statcast()
df = ler_e_limpar_dados(spark, CAMINHO_CSV)
assert len(df.columns) == len(esquema_statcast.fields)

df.write.format("delta").mode("overwrite").save(CAMINHO_DELTA)

print("Tabela Delta Statcast gravada em:", CAMINHO_DELTA)
df.show(5, truncate=False)

Tabela Delta criada com sucesso em /data/tabela_vendas_delta!


## UPDATE no Delta Lake (correção de `velocidade_media`)

**O que este bloco faz:** obtém a tabela Delta já gravada com `DeltaTable.forPath`, executa `update` com condição em `nome_jogador` e altera apenas `velocidade_media` via `F.lit`, e por fim filtra e exibe a linha do arremessador para conferência.

Cenário: após revisão de telemetria ou calibração de radar, corrige-se a **velocidade média** sem regravar o CSV inteiro. O predicado localiza o registro e o restante do *pipeline* permanece estável.

In [3]:
from delta.tables import DeltaTable
import pyspark.sql.functions as F

tabela_delta = DeltaTable.forPath(spark, CAMINHO_DELTA)

print("Ajustando velocidade_media de Webb, Logan após revisão de medição...")
tabela_delta.update(
    condition="nome_jogador = 'Webb, Logan'",
    set={"velocidade_media": F.lit(89.2)},
)

tabela_delta.toDF().filter(F.col("nome_jogador") == F.lit("Webb, Logan")).select(
    "nome_jogador", "velocidade_media", "total_arremessos", "strikeouts"
).show(truncate=False)

Atualizando o preço do Mouse...


26/04/18 14:59:31 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


+--------+--------+-----------+------+----------+----------+
|id_venda| produto|  categoria| valor|quantidade|      data|
+--------+--------+-----------+------+----------+----------+
|       1|Notebook|Eletrônicos|4500.0|         1|2026-04-18|
|       2|   Mouse| Acessórios| 135.0|         2|2026-04-18|
+--------+--------+-----------+------+----------+----------+



## INSERT no Delta Lake (novo jogador de teste)

**O que este bloco faz:** insere um registro sintético na tabela Delta para evidenciar operação de **INSERT** via Spark SQL e consulta o mesmo jogador para confirmação imediata da gravação.

**Teoria:** o `INSERT` cria uma nova versão da tabela no log transacional do Delta, mantendo rastreabilidade junto com `UPDATE` e `DELETE`.

In [ ]:
spark.sql(
    f"""
    INSERT INTO delta.`{CAMINHO_DELTA}`
    VALUES (
      'Pitcher, Teste',
      500,
      93.5,
      2400,
      0.210,
      110,
      10,
      30
    )
    """
)

spark.sql(
    f"""
    SELECT nome_jogador, velocidade_media, strikeouts
    FROM delta.`{CAMINHO_DELTA}`
    WHERE nome_jogador = 'Pitcher, Teste'
    """
).show(truncate=False)

## DELETE no Delta Lake (sanção ou linha inválida)

**O que este bloco faz:** reutiliza o objeto `tabela_delta`, chama `delete` com expressão no `nome_jogador` e consulta o DataFrame atual para mostrar como ficaram **Webb, Logan** e **Rodón, Carlos** após a remoção (a linha removida deixa de aparecer).

Cenário: **remoção por sanção**, duplicidade ou registro que não deve permanecer na camada curada. O Delta registra a operação no log para auditoria e *time travel*.

In [4]:
import pyspark.sql.functions as F

print("Removendo estatísticas de Rodón, Carlos (cenário de sanção / expurgo)...")
tabela_delta.delete(condition=F.expr("nome_jogador = 'Rodón, Carlos'"))

tabela_delta.toDF().filter(
    F.col("nome_jogador").isin("Webb, Logan", "Rodón, Carlos")
).select("nome_jogador", "velocidade_media", "strikeouts").show(truncate=False)

Deletando a venda do Notebook...
+--------+-------+----------+-----+----------+----------+
|id_venda|produto| categoria|valor|quantidade|      data|
+--------+-------+----------+-----+----------+----------+
|       2|  Mouse|Acessórios|135.0|         2|2026-04-18|
+--------+-------+----------+-----+----------+----------+



## Histórico de versões e time travel (auditoria Delta)

**O que este bloco faz:** projeta o histórico da tabela com `history()` (versão, instante, operação e parâmetros), depois lê a mesma tabela Delta na **versão 0** com `versionAsOf` para recuperar o estado inicial dos jogadores filtrados e confrontar com as mudanças aplicadas pelos `UPDATE` e `DELETE`.

In [5]:
import pyspark.sql.functions as F

print("Histórico de transações (Statcast):")
tabela_delta.history().select(
    "version", "timestamp", "operation", "operationParameters"
).show(truncate=False)

print("Time travel: versão 0 — Webb, Logan e Rodón, Carlos como na carga inicial")
df_versao_zero = (
    spark.read.format("delta")
    .option("versionAsOf", 0)
    .load(CAMINHO_DELTA)
    .filter(F.col("nome_jogador").isin("Webb, Logan", "Rodón, Carlos"))
    .select("nome_jogador", "velocidade_media", "strikeouts")
    .orderBy("nome_jogador")
)
df_versao_zero.show(truncate=False)

Histórico da Tabela:
+-------+--------------------+---------+--------------------+
|version|           timestamp|operation| operationParameters|
+-------+--------------------+---------+--------------------+
|      2|2026-04-18 15:03:...|   DELETE|{predicate -> ["(...|
|      1|2026-04-18 14:59:...|   UPDATE|{predicate -> ["(...|
|      0|2026-04-18 14:57:...|    WRITE|{mode -> Overwrit...|
+-------+--------------------+---------+--------------------+

Consultando a Versão 0 (Dados Originais):
+--------+--------+-----------+------+----------+----------+
|id_venda| produto|  categoria| valor|quantidade|      data|
+--------+--------+-----------+------+----------+----------+
|       1|Notebook|Eletrônicos|4500.0|         1|2026-04-18|
|       2|   Mouse| Acessórios| 120.0|         2|2026-04-18|
+--------+--------+-----------+------+----------+----------+

